# argmax-accuracy-eval — faded example 1: Balanced accuracy (macro-average of per-class recall)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `argmax-accuracy-eval`. Running the beacon reports progress on the `Eval: argmax accuracy` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Eval: argmax accuracy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`argmax-accuracy-eval`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "argmax-accuracy-eval"
DD_SUBTOPIC = "Eval: argmax accuracy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Balanced accuracy is the **mean over classes** of each class's recall (per-class top-1 accuracy). It de-biases the metric on imbalanced data. You compute per-class accuracy from `argmax` predictions, then average those class scores.

## Faded exercise 1

Implement `balanced_accuracy(logits, labels, num_classes)`. Predictions come from `logits.argmax(dim=-1)`. For each class `c`, compute the fraction of class-`c` examples that were predicted correctly (its recall), then return the mean of those per-class recalls as a Python float. You must complete the line that computes a single class's recall from the `correct` and `mask` boolean vectors.

**Fill in:** The recall for class c: the mean of `correct` over the rows where `labels == c`.

In [ ]:
def balanced_accuracy(logits, labels, num_classes):
    preds = logits.argmax(dim=-1)
    correct = (preds == labels)
    recalls = []
    for c in range(num_classes):
        mask = (labels == c)
        if mask.sum().item() == 0:
            continue
        class_recall = None  # TODO: mean of correct over rows where labels == c
        recalls.append(class_recall)
    return sum(recalls) / len(recalls)


def _test():
    t.manual_seed(1)
    C = 4
    logits = t.randn(40, C)
    labels = t.randint(0, C, (40,))
    got = balanced_accuracy(logits, labels, C)
    # independent reference
    preds = logits.argmax(dim=-1)
    ref_recalls = []
    for c in range(C):
        m = (labels == c)
        if m.sum().item() == 0:
            continue
        ref_recalls.append((preds[m] == labels[m]).float().mean().item())
    ref = sum(ref_recalls) / len(ref_recalls)
    assert isinstance(got, float)
    assert abs(got - ref) < 1e-6, (got, ref)
    assert 0.0 <= got <= 1.0


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def balanced_accuracy(logits, labels, num_classes):
    preds = logits.argmax(dim=-1)
    correct = (preds == labels)
    recalls = []
    for c in range(num_classes):
        mask = (labels == c)
        if mask.sum().item() == 0:
            continue
        class_recall = correct[mask].float().mean().item()
        recalls.append(class_recall)
    return sum(recalls) / len(recalls)
```
</details>